In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,when,count

In [0]:
spark=SparkSession.builder.appName("Data Cleaning").getOrCreate()
data=[
    (1, "Ravi", "Hyderabad", 25),
    (2, None, "Chennai", 32),
    (None, "Arun", "Hyderabad", 28),
    (4, "Meena", None, 30),
    (4, "Meena", None, 30),
    (5, "John", "Bangalore", -5)
]
columns = ["customer_id", "name", "city", "age"]
df=spark.createDataFrame(data,columns)
df.show()


+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|       NULL| Arun|Hyderabad| 28|
|          4|Meena|     NULL| 30|
|          4|Meena|     NULL| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+



In [0]:
duplicates_count=df.count()-df.dropDuplicates().count()
print("Duplicate rows:",duplicates_count)

Duplicate rows: 1


In [0]:
null_values=df.select([count(when(col(c).isNull(),c)).alias(c) for c in columns]).show()

+-----------+----+----+---+
|customer_id|name|city|age|
+-----------+----+----+---+
|          1|   1|   2|  0|
+-----------+----+----+---+



In [0]:
df.filter(col("age")<0).show()

+-----------+----+---------+---+
|customer_id|name|     city|age|
+-----------+----+---------+---+
|          5|John|Bangalore| -5|
+-----------+----+---------+---+



In [0]:
df_clean = df.filter(col("customer_id").isNotNull())

df_clean = df_clean.fillna({"name":"Unknown"})

df_clean.show()

+-----------+-------+---------+---+
|customer_id|   name|     city|age|
+-----------+-------+---------+---+
|          1|   Ravi|Hyderabad| 25|
|          2|Unknown|  Chennai| 32|
|          4|  Meena|     NULL| 30|
|          4|  Meena|     NULL| 30|
|          5|   John|Bangalore| -5|
+-----------+-------+---------+---+



In [0]:
df_clean=df_clean.dropDuplicates()
df_clean=df_clean.filter(col("age")>=0)
#after data cleaning
after_count=df_clean.count()
print("Rows After Cleaning:", after_count)
print("Cleaned Data")
df_clean.show()

Rows After Cleaning: 3
Cleaned Data
+-----------+-------+---------+---+
|customer_id|   name|     city|age|
+-----------+-------+---------+---+
|          1|   Ravi|Hyderabad| 25|
|          2|Unknown|  Chennai| 32|
|          4|  Meena|     NULL| 30|
+-----------+-------+---------+---+



In [0]:
#aggregation
print("customer by city")
city_count=df_clean.groupBy("city").count()
print(city_count.show())

customer by city
+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    1|
|  Chennai|    1|
|     NULL|    1|
+---------+-----+

None
